1. Create a class with 

In [1]:
import math

In [6]:
def EuropeanOption(S0, K, T, r, sigma, N, option_type='call', dividend=False):
    """
    Calculate the price of a European option using the Black-Scholes formula.

    Parameters:
    S0 : float
        Current stock price
    K : float
        Strike price
    T : float
        Time to maturity in years
    r : float
        Risk-free interest rate (annualized)
    sigma : float
        Volatility of the underlying stock (annualized)
    N : int
        Number of time steps for the binomial model 
    option_type : str, optional
        Type of option ('call' or 'put'), default is 'call'
    dividend : bool, optional
        Whether the stock pays dividends, default is False

    Returns:
    float
        Price of the European option
    """

    if not dividend:
        q = 0
    else:
        q = float(input("Enter the dividend yield (as a decimal): "))
    u = math.exp(sigma * math.sqrt(T/N))
    d = float(1/u)
    Pu = (math.exp((r-q) * T/N) - d)/(u - d)
    Pd = 1 - Pu
    prob = (Pd**N)
    S = S0 * d**N
    
    if option_type == 'call':
        payoff = max(0, S - K)
    else:
        payoff = max(0, K - S)

    for i in range(1, N+1):
        prob = prob * ((N-i+1)/i) * (Pu/Pd)
        S = S * u**2
        payoff = payoff + prob * max(0, S - K) if option_type == 'call' else payoff + prob * max(0, K - S)

    return payoff * math.exp(-r * T)


In [7]:
def AmericanOption(S0, K, T, r, sigma, N, option_type='call', dividend=False):
    """
    Calculate the price of an American option using the binomial model.

    Parameters:
    S0 : float
        Current stock price
    K : float
        Strike price
    T : float
        Time to maturity in years
    r : float
        Risk-free interest rate (annualized)
    sigma : float
        Volatility of the underlying stock (annualized)
    N : int
        Number of time steps for the binomial model
    option_type : str, optional
        Type of option ('call' or 'put'), default is 'call'

    Returns:
    float
        Price of the American option
    """

    if not dividend:
        q = 0
    else:
        q = float(input("Enter the dividend yield (as a decimal): "))
    dt = T / N
    u = math.exp(sigma * math.sqrt(dt))
    d = 1 / u
    p = (math.exp((r - q) * dt) - d) / (u - d)
    discount = math.exp(-r * dt)

    V = [0.0] * (N + 1)
    for i in range(N + 1):
        St = S0 * (u**i) * (d**(N - i))
        if option_type == 'call':
            V[i] = max(0, St - K)
        else:
            V[i] = max(0, K - St)

    for j in range(N - 1, -1, -1):
        for i in range(j + 1):
            St = S0 * (u**i) * (d**(j - i))
            if option_type == 'call':
                EV = max(0, St - K)
            else:
                EV = max(0, K - St)

            CV = discount * (p * V[i + 1] + (1 - p) * V[i])
            V[i] = max(EV, CV)

    return V[0]
    